# Week 5 Day 3 - Microsoft Agent Framework exercise

In [ ]:
# The framework prints a couple of "experimental" notices on import; quiet them.
import warnings
warnings.filterwarnings("ignore", message=r".*experimental.*")
import subprocess
from pathlib import Path

from dotenv import load_dotenv
from agent_framework import Agent, MCPStdioTool
from agent_framework.openai import OpenAIChatClient

import board

load_dotenv(override=True)

In [ ]:
MODEL = "gpt-5.4-mini"

client = OpenAIChatClient(model=MODEL)

In [ ]:
def show_todos() -> list[dict]:
    """List every todo on the board. A goal has parent_id None; a step has parent_id set to its goal's id."""
    return board.list_todos()

def plan_steps(goal_id: int, steps: list[str]) -> dict:
    """Break a goal into an ordered checklist of steps on the board. Pass the goal's id and a short list of step descriptions."""
    return {"goal_id": goal_id, "step_ids": [board.add_step(goal_id, step) for step in steps]}

def complete_task(task_id: int, result: str) -> dict:
    """Mark a todo (a step or the goal) with this id as done and record a short result summary."""
    board.complete_todo(task_id, result)
    return {"task_id": task_id, "status": "done"}

In [ ]:
workspace = Path("workspace").resolve()   # the only folder the agent may touch
workspace.mkdir(exist_ok=True)


class FilesystemMCP(MCPStdioTool):
    """The filesystem server with its stderr sent to DEVNULL and its working
    directory set to the workspace, so file names resolve there and it runs
    cleanly from a Jupyter kernel on Windows."""

    def get_mcp_client(self):
        from mcp.client.stdio import StdioServerParameters, stdio_client

        params = StdioServerParameters(command=self.command, args=self.args, env=self.env, cwd=str(workspace))
        return stdio_client(server=params, errlog=subprocess.DEVNULL)


filesystem = FilesystemMCP(
    name="filesystem",
    command="npx",
    args=["-y", "@modelcontextprotocol/server-filesystem", str(workspace)],
)

In [ ]:
INSTRUCTIONS = """
You are a careful worker with a shared todo board and a set of file tools.

Take the pending goal and see it through. Begin by laying out a short plan: the handful of concrete steps the work itself breaks down into, added to the board under the goal. Then carry them out with your file tools, marking each step done as you finish it. Once the steps are all done, close the goal. Your files live in the single folder your tools are allowed to use.
"""

worker = Agent(
    client=client,
    instructions=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

board.reset_board()
goal_id = board.add_goal("Write a short haiku about Madrid into madrid.txt.")
board.claim_todo(goal_id)

async with filesystem:
    await worker.run("Please work the pending goal on the board.")
board.show_board()

## Swap the model: Gemini through its OpenAI-compatible endpoint

In [ ]:
import os

from agent_framework.openai import OpenAIChatCompletionClient

# Same agent, different model: only the client changes.
# Gemini only speaks the Chat Completions API, so we use the Chat Completions client.
client = OpenAIChatCompletionClient(
    model="gemini-2.5-flash",
    api_key=os.environ["GOOGLE_API_KEY"],
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)

In [ ]:
worker = Agent(
    client=client,
    instructions=INSTRUCTIONS,
    tools=[show_todos, plan_steps, complete_task, filesystem],
)

board.reset_board()
goal_id = board.add_goal("Write a short haiku about Madrid into madrid.txt.")
board.claim_todo(goal_id)

async with filesystem:
    await worker.run("Please work the pending goal on the board.")
board.show_board()